In [ ]:
%pip install pandas nltk langdetect simplemma


In [ ]:
from pathlib import Path

import pandas as pd

cwd = Path.cwd()
raiz_projeto = cwd.parent if cwd.name == "Limpeza_Preparacao" else cwd

pasta_brutos = raiz_projeto / "DadosBrutos"
pasta_limpos = raiz_projeto / "DadosLimpos"
pasta_limpos.mkdir(exist_ok=True)

caminho_jogos = pasta_brutos / "jogos_steam.csv"
caminho_reviews = pasta_brutos / "steam_reviews.csv"

jogos = pd.read_csv(caminho_jogos, low_memory=False)
reviews = pd.read_csv(caminho_reviews, low_memory=False)

descricao_vazia = jogos["description"].isna() | jogos["description"].astype("string").str.strip().eq("")
appids_removidos = set(jogos.loc[descricao_vazia, "appid"])

jogos_limpo = jogos.loc[~jogos["appid"].isin(appids_removidos)].copy()
reviews_limpo = reviews.loc[~reviews["appid"].isin(appids_removidos)].copy()

jogos_limpo.to_csv(pasta_limpos / "jogos_steam.csv", index=False)
reviews_limpo.to_csv(pasta_limpos / "steam_reviews.csv", index=False)

print(f"Jogos antes: {len(jogos):,}")
print(f"Reviews antes: {len(reviews):,}")
print(f"Jogos removidos por descricao vazia: {len(appids_removidos):,}")
print(f"Reviews removidas dos jogos sem descricao: {len(reviews) - len(reviews_limpo):,}")
print(f"Jogos depois: {len(jogos_limpo):,}")
print(f"Reviews depois: {len(reviews_limpo):,}")


In [ ]:
import unicodedata


def tem_letra_nao_latina(texto):
    if pd.isna(texto):
        return False
    # Normaliza variantes tipograficas e preserva acentos, simbolos e pontuacao.
    texto = unicodedata.normalize("NFKC", str(texto))
    return any(
        caractere.isalpha() and "LATIN" not in unicodedata.name(caractere, "")
        for caractere in texto
    )


nome_nao_latino = jogos_limpo["name"].map(tem_letra_nao_latina)
descricao_nao_latina = jogos_limpo["description"].map(tem_letra_nao_latina)
appids_nao_latinos = set(
    jogos_limpo.loc[nome_nao_latino | descricao_nao_latina, "appid"]
)

quantidade_reviews_antes = len(reviews_limpo)
jogos_limpo = jogos_limpo.loc[~jogos_limpo["appid"].isin(appids_nao_latinos)].copy()
reviews_limpo = reviews_limpo.loc[~reviews_limpo["appid"].isin(appids_nao_latinos)].copy()

jogos_limpo.to_csv(pasta_limpos / "jogos_steam.csv", index=False)
reviews_limpo.to_csv(pasta_limpos / "steam_reviews.csv", index=False)

print(f"Jogos removidos por letras nao latinas: {len(appids_nao_latinos):,}")
print(f"Reviews removidas desses jogos: {quantidade_reviews_antes - len(reviews_limpo):,}")
print(f"Jogos restantes: {len(jogos_limpo):,}")
print(f"Reviews restantes: {len(reviews_limpo):,}")


In [ ]:
import re
from html.parser import HTMLParser


class ExtratorTextoHTML(HTMLParser):
    blocos = {
        "p", "div", "br", "hr", "li", "ul", "ol", "table", "tr", "td",
        "th", "h1", "h2", "h3", "h4", "h5", "h6", "section", "blockquote",
    }

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.partes = []
        self.ignorar = 0

    def handle_starttag(self, tag, attrs):
        if tag in {"script", "style"}:
            self.ignorar += 1
        if tag in self.blocos and not self.ignorar:
            self.partes.append(" ")

    def handle_endtag(self, tag):
        if tag in {"script", "style"}:
            self.ignorar = max(0, self.ignorar - 1)
        if tag in self.blocos and not self.ignorar:
            self.partes.append(" ")

    def handle_data(self, data):
        if not self.ignorar:
            self.partes.append(data)


def limpar_texto(texto):
    if pd.isna(texto):
        return ""
    parser = ExtratorTextoHTML()
    parser.feed(str(texto))
    parser.close()
    texto = unicodedata.normalize("NFKC", "".join(parser.partes))
    # Remove caracteres invisiveis sem retirar acentos, pontuacao ou negacoes.
    texto = texto.translate(dict.fromkeys(map(ord, "\u200b\ufeff\u00ad")))
    texto = "".join(
        c for c in texto
        if unicodedata.category(c) != "Cc" or c.isspace()
    )
    return re.sub(r"\s+", " ", texto).strip()


jogos_limpo["description"] = jogos_limpo["description"].map(limpar_texto)
reviews_limpo["review"] = reviews_limpo["review"].map(limpar_texto)

appids_vazios_apos_limpeza = set(
    jogos_limpo.loc[jogos_limpo["description"].eq(""), "appid"]
)
reviews_antes_limpeza = len(reviews_limpo)
jogos_limpo = jogos_limpo.loc[
    ~jogos_limpo["appid"].isin(appids_vazios_apos_limpeza)
].copy()
reviews_limpo = reviews_limpo.loc[
    ~reviews_limpo["appid"].isin(appids_vazios_apos_limpeza)
].copy()

jogos_limpo.to_csv(pasta_limpos / "jogos_steam.csv", index=False)
reviews_limpo.to_csv(pasta_limpos / "steam_reviews.csv", index=False)

print(f"Jogos removidos por descricao vazia apos limpeza: {len(appids_vazios_apos_limpeza):,}")
print(f"Reviews removidas desses jogos: {reviews_antes_limpeza - len(reviews_limpo):,}")
print(f"Reviews sem texto (mantidas): {reviews_limpo['review'].eq('').sum():,}")
print(f"Jogos restantes: {len(jogos_limpo):,}")
print(f"Reviews restantes: {len(reviews_limpo):,}")


In [ ]:
import json
from nltk.tokenize import TweetTokenizer

tokenizador = TweetTokenizer(preserve_case=True, reduce_len=False, strip_handles=False)


def tokenizar_texto(texto):
    if pd.isna(texto):
        return []
    return tokenizador.tokenize(str(texto))


jogos_limpo["description_tokens"] = jogos_limpo["description"].map(tokenizar_texto)
reviews_limpo["review_tokens"] = reviews_limpo["review"].map(tokenizar_texto)

# JSON permite recuperar as listas com json.loads ao reler os CSVs.
for dados, coluna_tokens, nome_arquivo in [
    (jogos_limpo, "description_tokens", "jogos_steam.csv"),
    (reviews_limpo, "review_tokens", "steam_reviews.csv"),
]:
    dados.assign(**{
        coluna_tokens: dados[coluna_tokens].map(
            lambda tokens: json.dumps(tokens, ensure_ascii=False)
        )
    }).to_csv(pasta_limpos / nome_arquivo, index=False)

print(f"Tokens nas descricoes: {jogos_limpo['description_tokens'].map(len).sum():,}")
print(f"Tokens nas reviews: {reviews_limpo['review_tokens'].map(len).sum():,}")
print(f"Reviews sem tokens: {reviews_limpo['review_tokens'].map(len).eq(0).sum():,}")

display(jogos_limpo[["appid", "description", "description_tokens"]].head(3))
display(reviews_limpo[["appid", "review", "review_tokens"]].head(3))


In [ ]:
import json
import unicodedata

apostrofos = str.maketrans({"\u2018": "'", "\u2019": "'", "\u02bc": "'"})


def normalizar_tokens(tokens):
    return [
        unicodedata.normalize(
            "NFKC", unicodedata.normalize("NFKC", token).translate(apostrofos).lower()
        )
        for token in tokens
    ]


jogos_limpo["description_tokens_normalizados"] = jogos_limpo["description_tokens"].map(
    normalizar_tokens
)
reviews_limpo["review_tokens_normalizados"] = reviews_limpo["review_tokens"].map(
    normalizar_tokens
)

# Serializa ambas as listas em JSON, mantendo listas nos DataFrames em memoria.
for dados, coluna_tokens, nome_arquivo in [
    (jogos_limpo, "description_tokens", "jogos_steam.csv"),
    (reviews_limpo, "review_tokens", "steam_reviews.csv"),
]:
    colunas_listas = [coluna_tokens, f"{coluna_tokens}_normalizados"]
    dados.assign(**{
        coluna: dados[coluna].map(lambda tokens: json.dumps(tokens, ensure_ascii=False))
        for coluna in colunas_listas
    }).to_csv(pasta_limpos / nome_arquivo, index=False)

print(f"Descricoes normalizadas: {len(jogos_limpo):,}")
print(f"Reviews normalizadas: {len(reviews_limpo):,}")

display(jogos_limpo[
    ["appid", "description_tokens", "description_tokens_normalizados"]
].head(3))
display(reviews_limpo[
    ["appid", "review_tokens", "review_tokens_normalizados"]
].head(3))


In [ ]:
import nltk
from nltk.corpus import stopwords
from langdetect import DetectorFactory, detect_langs, LangDetectException

try:
    stopwords.fileids()
except LookupError:
    nltk.download("stopwords", raise_on_error=True)

DetectorFactory.seed = 0
idiomas_stopwords = {
    "en": "english", "pt": "portuguese", "es": "spanish",
    "fr": "french", "it": "italian", "de": "german", "nl": "dutch",
}
# Preserva negacoes para nao inverter o sentido, especialmente nas reviews.
negacoes = set(normalizar_tokens([
    "no", "not", "nor", "never", "neither", "nothing", "nobody", "without",
    "cannot", "n't", "n\u00e3o", "nem", "nunca", "jamais", "sem",
    "nada", "ningu\u00e9m", "ninguem", "nao", "ni", "sin", "nadie",
    "ning\u00fan", "ninguno", "ninguna", "tampoco",
    "ne", "n'", "pas", "plus", "rien", "aucun", "aucune", "sans",
    "non", "mai", "nessuno", "nessuna", "niente", "senza",
    "nicht", "kein", "keine", "keinen", "keinem", "keiner", "keines",
    "nie", "niemals", "niemand", "nichts", "ohne",
    "niet", "geen", "nooit", "niemand", "niets", "zonder",
]))
stopwords_por_idioma = {
    codigo: {
        palavra for palavra in normalizar_tokens(stopwords.words(nome))
        if palavra not in negacoes and not palavra.endswith("n't")
    }
    for codigo, nome in idiomas_stopwords.items()
}


def detectar_idioma(texto):
    if pd.isna(texto):
        return "indeterminado"
    # A detecao e heuristica; textos curtos e resultados incertos ficam intactos.
    amostra = str(texto)[:1500]
    if len(amostra.split()) < 5 or sum(c.isalpha() for c in amostra) < 20:
        return "indeterminado"
    try:
        candidato = detect_langs(amostra)[0]
    except LangDetectException:
        return "indeterminado"
    return candidato.lang if candidato.prob >= 0.90 else "indeterminado"


def remover_stopwords(tokens, idioma):
    palavras = stopwords_por_idioma.get(idioma, set())
    return [token for token in tokens if token not in palavras]


for dados, texto, nome_arquivo in [
    (jogos_limpo, "description", "jogos_steam.csv"),
    (reviews_limpo, "review", "steam_reviews.csv"),
]:
    coluna_idioma = f"{texto}_idioma"
    coluna_entrada = f"{texto}_tokens_normalizados"
    coluna_saida = f"{texto}_tokens_sem_stopwords"
    dados[coluna_idioma] = dados[texto].map(detectar_idioma)
    dados[coluna_saida] = [
        remover_stopwords(tokens, idioma)
        for tokens, idioma in zip(dados[coluna_entrada], dados[coluna_idioma])
    ]

    colunas_listas = [f"{texto}_tokens", coluna_entrada, coluna_saida]
    dados.assign(**{
        coluna: dados[coluna].map(lambda tokens: json.dumps(tokens, ensure_ascii=False))
        for coluna in colunas_listas
    }).to_csv(pasta_limpos / nome_arquivo, index=False)

    removidos = dados[coluna_entrada].map(len).sum() - dados[coluna_saida].map(len).sum()
    sem_lista = ~dados[coluna_idioma].isin(stopwords_por_idioma)
    print(f"{texto}: {removidos:,} stopwords removidas")
    print(f"{texto}: {sem_lista.sum():,} textos mantidos sem filtragem por idioma")
    display(dados[["appid", coluna_idioma, coluna_entrada, coluna_saida]].head(3))


In [ ]:
import json
from functools import lru_cache
from nltk.stem.snowball import SnowballStemmer

stemmers = {
    codigo: SnowballStemmer(nome)
    for codigo, nome in idiomas_stopwords.items()
    if nome in SnowballStemmer.languages
}


@lru_cache(maxsize=100000)
def aplicar_stemming(token, idioma):
    stemmer = stemmers.get(idioma)
    if stemmer is None or token in negacoes or not token.isalpha():
        return token
    return stemmer.stem(token)


# As duas alternativas partem dos tokens sem stopwords.
jogos_stemming = jogos_limpo.copy()
reviews_stemming = reviews_limpo.copy()
pasta_stemming = pasta_limpos / "stemming"
pasta_stemming.mkdir(parents=True, exist_ok=True)

for dados, texto, nome_arquivo in [
    (jogos_stemming, "description", "jogos_steam.csv"),
    (reviews_stemming, "review", "steam_reviews.csv"),
]:
    entrada = f"{texto}_tokens_sem_stopwords"
    saida = f"{texto}_tokens_stemming"
    idiomas = dados[f"{texto}_idioma"]
    dados[saida] = [
        [aplicar_stemming(token, idioma) for token in tokens]
        for tokens, idioma in zip(dados[entrada], idiomas)
    ]
    colunas_listas = [
        f"{texto}_tokens", f"{texto}_tokens_normalizados", entrada, saida,
    ]
    dados.assign(**{
        coluna: dados[coluna].map(lambda tokens: json.dumps(tokens, ensure_ascii=False))
        for coluna in colunas_listas
    }).to_csv(pasta_stemming / nome_arquivo, index=False)
    print(f"{texto}: {len(dados):,} registros salvos em {pasta_stemming / nome_arquivo}")
    print(f"Idioma sem stemmer (tokens mantidos): {(~idiomas.isin(stemmers)).sum():,}")
    display(dados[["appid", f"{texto}_idioma", entrada, saida]].head(3))


In [ ]:
import json
from functools import lru_cache
import simplemma

idiomas_lematizacao = {"en", "pt", "es", "fr", "it", "de", "nl"}


@lru_cache(maxsize=100000)
def lematizar_token(token, idioma):
    if idioma not in idiomas_lematizacao or token in negacoes or not token.isalpha():
        return token
    # Lematizacao por dicionario: nao desambigua pelo contexto da frase.
    return simplemma.lemmatize(token, lang=idioma)


jogos_lematizados = jogos_limpo.copy()
reviews_lematizadas = reviews_limpo.copy()
pasta_lematizacao = pasta_limpos / "lematizacao"
pasta_lematizacao.mkdir(parents=True, exist_ok=True)

for dados, texto, nome_arquivo in [
    (jogos_lematizados, "description", "jogos_steam.csv"),
    (reviews_lematizadas, "review", "steam_reviews.csv"),
]:
    entrada = f"{texto}_tokens_sem_stopwords"
    saida = f"{texto}_tokens_lematizados"
    idiomas = dados[f"{texto}_idioma"]
    dados[saida] = [
        [lematizar_token(token, idioma) for token in tokens]
        for tokens, idioma in zip(dados[entrada], idiomas)
    ]
    colunas_listas = [
        f"{texto}_tokens", f"{texto}_tokens_normalizados", entrada, saida,
    ]
    dados.assign(**{
        coluna: dados[coluna].map(lambda tokens: json.dumps(tokens, ensure_ascii=False))
        for coluna in colunas_listas
    }).to_csv(pasta_lematizacao / nome_arquivo, index=False)
    print(f"{texto}: {len(dados):,} registros salvos em {pasta_lematizacao / nome_arquivo}")
    print(f"Idioma sem lematizador (tokens mantidos): {(~idiomas.isin(idiomas_lematizacao)).sum():,}")
    display(dados[["appid", f"{texto}_idioma", entrada, saida]].head(3))


In [ ]:
from IPython.display import display

# Seleciona ate 10 jogos mantidos e a primeira review disponivel de cada um.
ids_pipeline = jogos_limpo["appid"].drop_duplicates().head(10)
amostra_jogos = jogos.loc[
    jogos["appid"].isin(ids_pipeline), ["appid", "name", "description"]
].drop_duplicates("appid")
amostra_reviews = (
    reviews.loc[reviews["appid"].isin(ids_pipeline), ["appid", "review"]]
    .drop_duplicates("appid", keep="first")
    .assign(tem_review=True)
)
amostra_pipeline = (
    amostra_jogos
    .merge(amostra_reviews, on="appid", how="left", validate="one_to_one")
    .assign(tem_review=lambda df: df["tem_review"].eq(True))
    .rename(columns={"description": "descricao_original", "review": "review_original"})
)


def etapa_filtros(df):
    return df.assign(
        descricao_valida=~(
            df["descricao_original"].isna()
            | df["descricao_original"].astype("string").str.strip().eq("")
        ),
        letras_latinas=~(
            df["name"].map(tem_letra_nao_latina)
            | df["descricao_original"].map(tem_letra_nao_latina)
        ),
    )


def etapa_textos(df, sufixo_entrada, sufixo_saida, funcao):
    return df.assign(**{
        f"{campo}_{sufixo_saida}": df[f"{campo}_{sufixo_entrada}"].map(funcao)
        for campo in ("descricao", "review")
    })


def etapa_por_idioma(df, sufixo_saida, funcao):
    return df.assign(**{
        f"{campo}_{sufixo_saida}": [
            funcao(tokens, idioma)
            for tokens, idioma in zip(
                df[f"{campo}_normalizados"], df[f"{campo}_idioma"]
            )
        ]
        for campo in ("descricao", "review")
    })


def etapa_alternativa(df, sufixo_saida, funcao):
    return df.assign(**{
        f"{campo}_{sufixo_saida}": [
            [funcao(token, idioma) for token in tokens]
            for tokens, idioma in zip(
                df[f"{campo}_sem_stopwords"], df[f"{campo}_idioma"]
            )
        ]
        for campo in ("descricao", "review")
    })


resultado_pipeline = (
    amostra_pipeline
    .pipe(etapa_filtros)
    .pipe(etapa_textos, "original", "limpa", limpar_texto)
    .pipe(etapa_textos, "limpa", "idioma", detectar_idioma)
    .pipe(etapa_textos, "limpa", "tokens", tokenizar_texto)
    .pipe(etapa_textos, "tokens", "normalizados", normalizar_tokens)
    .pipe(etapa_por_idioma, "sem_stopwords", remover_stopwords)
    .pipe(etapa_alternativa, "stemming", aplicar_stemming)
    .pipe(etapa_alternativa, "lematizados", lematizar_token)
)

etapas_pipeline = {
    "Texto original": "original",
    "Limpeza": "limpa",
    "Idioma estimado": "idioma",
    "Tokenizacao": "tokens",
    "Normalizacao": "normalizados",
    "Sem stopwords": "sem_stopwords",
    "Stemming (alternativa)": "stemming",
    "Lematizacao (alternativa)": "lematizados",
}
comparacao_pipeline = pd.concat([
    resultado_pipeline[
        ["appid", "name", f"descricao_{sufixo}", f"review_{sufixo}"]
    ].rename(columns={
        f"descricao_{sufixo}": "descricao",
        f"review_{sufixo}": "review",
    }).assign(etapa=etapa)
    for etapa, sufixo in etapas_pipeline.items()
], ignore_index=True)

print(f"Pipeline demonstrativo: {len(resultado_pipeline)} jogos mantidos pela limpeza.")
print("Reviews: primeira linha de cada jogo; jogos sem review permanecem na amostra.")
print("Previa limitada a 160 caracteres/20 tokens; resultados completos nos DataFrames.")
with pd.option_context("display.max_colwidth", 160, "display.max_seq_items", 20):
    display(resultado_pipeline[
        ["appid", "name", "descricao_valida", "letras_latinas", "tem_review"]
    ])
    for appid, nome in resultado_pipeline[["appid", "name"]].itertuples(index=False, name=None):
        print(f"{appid} - {nome}")
        display(
            comparacao_pipeline.loc[comparacao_pipeline["appid"].eq(appid)]
            .set_index("etapa")[["descricao", "review"]]
        )
